# ATN Training — Cloud GPU Edition
### Works on: Kaggle (free P100) · Google Colab (free T4) · RunPod · Vast.ai

**How to use:**
1. Upload your `atn_train/` and `atn_val/` image folders as a Kaggle Dataset
   *(or mount Google Drive on Colab — instructions in Cell 0)*
2. Set `PLATFORM` in Cell 1 to `'kaggle'` or `'colab'`
3. Run all cells — training takes ~5–10 min/epoch on a free GPU
4. Download `reface_atn.pth` from the output section when done

## Cell 0 — Platform Setup Instructions

### Kaggle (recommended — free, no session limit, P100 GPU)
1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) → **New Dataset**
2. Upload a `.zip` of your `atn_train/` folder → name it `atn-train-data`
3. Upload a `.zip` of your `atn_val/` folder → name it `atn-val-data`
4. Create a new **Notebook** → add both datasets via **+ Add Data**
5. Set accelerator to **GPU P100** in Settings → right panel
6. Paste this notebook and run

### Google Colab (free T4, 12h session)
1. Open [colab.research.google.com](https://colab.research.google.com)
2. Runtime → Change runtime type → **T4 GPU**
3. Upload images to Google Drive in folders `atn_train/` and `atn_val/`
4. Set `PLATFORM = 'colab'` in Cell 1 — it will auto-mount Drive

### RunPod / Vast.ai (paid, ~$0.20/hr, most control)
1. Rent a pod with PyTorch image (RTX 3090 recommended)
2. `scp` your data folders to the pod
3. Set `PLATFORM = 'runpod'` in Cell 1 and set paths manually

In [ ]:
# Cell 1 - Setup (Kaggle-first paths, hyperparams, deps)
import os, sys, json, gc, math, time, random, hashlib, subprocess
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Kaggle default (change only if you intentionally run elsewhere)
PLATFORM = 'kaggle'

# Install missing packages on first run
def _pip_install(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

try:
    from facenet_pytorch import InceptionResnetV1
except ModuleNotFoundError:
    # Avoid touching Kaggle's core dependency set (numpy/scipy/pandas versions).
    _pip_install('facenet-pytorch', '--no-deps')
    from facenet_pytorch import InceptionResnetV1

# Do not import/install scikit-image on Kaggle to avoid numpy/scipy resolver conflicts.
# We use PSNR in Cell 9 instead of SSIM.

# AMP compatibility across torch versions used on Kaggle
try:
    from torch.amp import autocast as _autocast_new, GradScaler as _GradScalerNew
    AMP_STYLE = 'torch.amp'
except ImportError:
    from torch.cuda.amp import autocast as _autocast_old, GradScaler as _GradScalerOld
    AMP_STYLE = 'torch.cuda.amp'

def amp_autocast(enabled):
    if AMP_STYLE == 'torch.amp':
        return _autocast_new('cuda', enabled=enabled)
    return _autocast_old(enabled=enabled)

def make_grad_scaler(enabled):
    if AMP_STYLE == 'torch.amp':
        return _GradScalerNew('cuda', enabled=enabled)
    return _GradScalerOld(enabled=enabled)

# -- Paths per platform --------------------------------------------------------------
if PLATFORM == 'kaggle':
    TRAIN_DIR = Path('/kaggle/input/datasets/aryanrudrpandit/atn-train-data')
    VAL_DIR   = Path('/kaggle/input/datasets/aryanrudrpandit/atn-val-data')
    OUT_DIR   = Path('/kaggle/working')
else:
    raise ValueError(f'Unknown PLATFORM: {PLATFORM}')

OUTPUT_PATH    = OUT_DIR / 'reface_atn.pth'
BEST_PATH      = OUT_DIR / 'reface_atn_best.pth'
CHECKPOINT_DIR = OUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Detector checkpoint discovery (Kaggle-friendly + recursive fallback)
deepsafe_candidates = [
    OUT_DIR / 'deepsafe.pth',
    Path('/kaggle/input/deepfake-detection-models/deepsafe.pth'),
    Path('/kaggle/input/deepsafe-model/deepsafe.pth'),
    Path('/kaggle/input/afs-models/deepsafe.pth'),
]
DEEPSAFE_PTH = next((p for p in deepsafe_candidates if p.exists()), None)

if DEEPSAFE_PTH is None and PLATFORM == 'kaggle' and Path('/kaggle/input').exists():
    for p in Path('/kaggle/input').rglob('deepsafe.pth'):
        DEEPSAFE_PTH = p
        break

# -- Hyperparameters -----------------------------------------------------------------
EPOCHS       = 40
LR           = 3e-4
EPSILON      = 0.20
LAMBDA_X     = 1.0
LAMBDA_Y     = 50.0
LAMBDA_DET   = 10.0
MARGIN       = 0.30
IMG_SIZE     = 224
BATCH_SIZE   = 64
NUM_WORKERS  = 4
CACHE_REFRESH_EVERY = 10

DETECTOR_ENABLED = DEEPSAFE_PTH is not None
if not DETECTOR_ENABLED:
    print('WARNING: deepsafe.pth not found. Detector-aware loss disabled for this run.')
    LAMBDA_DET = 0.0

# -- Device / CUDA compatibility -----------------------------------------------------
HAS_CUDA = torch.cuda.is_available()
CUDA_COMPATIBLE = False
CUDA_REASON = 'CUDA not available'
GPU_NAME = 'N/A'
GPU_CAP = (0, 0)

if HAS_CUDA:
    try:
        GPU_NAME = torch.cuda.get_device_name(0)
        GPU_CAP = torch.cuda.get_device_capability(0)
        # Current Kaggle torch build in your run supports sm_70+, but P100 is sm_60.
        if GPU_CAP[0] >= 7:
            CUDA_COMPATIBLE = True
            CUDA_REASON = 'CUDA compatible'
        else:
            CUDA_REASON = f'Unsupported GPU capability sm_{GPU_CAP[0]}{GPU_CAP[1]} for this torch build'
    except Exception as ex:
        CUDA_REASON = f'CUDA probe failed: {ex}'

if CUDA_COMPATIBLE:
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
USE_AMP = DEVICE.type == 'cuda'

print('='*55)
print(f'  Platform  : {PLATFORM}')
print(f'  Device    : {DEVICE}')
print(f'  AMP API   : {AMP_STYLE}')
print(f'  CUDA state: {CUDA_REASON}')
print(f'  GPU name  : {GPU_NAME}')
print(f'  GPU cap   : {GPU_CAP}')
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'  VRAM      : {props.total_memory/1e9:.1f} GB')
print(f'  AMP       : {USE_AMP}')
print(f'  Epsilon   : {EPSILON}')
print(f'  λx / λy   : {LAMBDA_X} / {LAMBDA_Y}')
print(f'  λdet      : {LAMBDA_DET}')
print(f'  Epochs    : {EPOCHS}')
print(f'  LR        : {LR}')
print(f'  Train dir : {TRAIN_DIR}')
print(f'  Val dir   : {VAL_DIR}')
print(f'  Output    : {OUTPUT_PATH}')
print(f'  Detector  : {DEEPSAFE_PTH if DETECTOR_ENABLED else "DISABLED"}')
print('='*55)
assert TRAIN_DIR.exists(), f'TRAIN_DIR not found: {TRAIN_DIR}. Add your train dataset in Kaggle Add Data.'
assert VAL_DIR.exists(),   f'VAL_DIR not found: {VAL_DIR}. Add your val dataset in Kaggle Add Data.'
print('Paths verified ✓')


CalledProcessError: Command '['d:\\deepfake_detection\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', 'torch==2.3.1', 'torchvision==0.18.1', 'pillow==10.2.0', 'scikit-image', 'tqdm']' returned non-zero exit status 1.

In [ ]:
# Cell 2 - Dataset Preview
def list_images(d):
    exts = {'.jpg','.jpeg','.png','.bmp','.webp'}
    return [p for p in Path(d).rglob('*') if p.suffix.lower() in exts]

train_images = list_images(TRAIN_DIR)
val_images   = list_images(VAL_DIR)
print(f'Train: {len(train_images)} images')
print(f'Val  : {len(val_images)} images')

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for ax, p in zip(axes.flatten(), train_images[:9]):
    img = Image.open(p).convert('RGB')
    ax.imshow(img); ax.set_title(f'{img.size[0]}x{img.size[1]}'); ax.axis('off')
for ax in axes.flatten()[9:]: ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# Cell 3 - Dataset & DataLoader
class FaceDataset(Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = paths, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.tf(img), i   # also return index so cache lookup is exact

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])
# Used for caching and validation — no stochastic augmentation
clean_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])
val_tf = clean_tf

# shuffle=True for training (augmented); DataLoader for cache uses clean_tf + shuffle=False
train_loader = DataLoader(FaceDataset(train_images, train_tf),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)

# Separate loader for building the embedding cache — deterministic order, no augmentation
cache_loader = DataLoader(FaceDataset(train_images, clean_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

val_loader = DataLoader(FaceDataset(val_images, val_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches/epoch : {len(train_loader)}')
b, _ = next(iter(train_loader))
print(f'First batch shape   : {tuple(b.shape)}')


In [ ]:
# Cell 4 - ATN Architecture
class ResidualBlock(nn.Module):
    def __init__(self, ch, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.GELU(),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch),
        )
    def forward(self, x):
        return F.gelu(self.net(x) + x)

class DownBlock(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.c = nn.Sequential(
            nn.Conv2d(ic, oc, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(oc), nn.GELU(), ResidualBlock(oc)
        )
    def forward(self, x):
        return self.c(x)

class UpBlock(nn.Module):
    def __init__(self, ic, sc, oc):
        super().__init__()
        self.up = nn.ConvTranspose2d(ic, oc, 2, stride=2)
        self.fuse = nn.Sequential(
            nn.Conv2d(oc + sc, oc, 3, padding=1, bias=False),
            nn.BatchNorm2d(oc), nn.GELU(), ResidualBlock(oc)
        )
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.fuse(torch.cat([x, skip], dim=1))

class ResidualUNetATN(nn.Module):
    def __init__(self, eps=0.05, base_ch=32, max_ch=256):
        super().__init__()
        c1 = base_ch
        c2 = min(max_ch, c1 * 2)
        c3 = min(max_ch, c2 * 2)
        c4 = min(max_ch, c3 * 2)

        self.eps = eps
        self.stem = nn.Sequential(nn.Conv2d(3, c1, 3, padding=1, bias=False), nn.BatchNorm2d(c1), nn.GELU())
        self.down1 = DownBlock(c1, c2)
        self.down2 = DownBlock(c2, c3)
        self.down3 = DownBlock(c3, c4)
        self.down4 = DownBlock(c4, c4)

        # Bottleneck with slight dropout to prevent lazy shortcuts
        self.bottleneck = nn.Sequential(
            ResidualBlock(c4, dropout=0.1),
            ResidualBlock(c4, dropout=0.1),
        )

        self.up1 = UpBlock(c4, c4, c4)
        self.up2 = UpBlock(c4, c3, c3)
        self.up3 = UpBlock(c3, c2, c2)
        self.up4 = UpBlock(c2, c1, c1)
        self.out = nn.Conv2d(c1, 3, 3, padding=1)

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.down1(s0)
        s2 = self.down2(s1)
        s3 = self.down3(s2)
        s4 = self.down4(s3)
        b = self.bottleneck(s4)
        d1 = self.up1(b, s3)
        d2 = self.up2(d1, s2)
        d3 = self.up3(d2, s1)
        d4 = self.up4(d3, s0)
        # eps * tanh: bounded perturbation, smooth gradients
        return self.eps * torch.tanh(self.out(d4))

# Auto low-VRAM profile for 16 GB class GPUs (e.g., T4/P100 sessions under pressure).
if DEVICE.type == 'cuda':
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
else:
    total_vram_gb = 0.0

LOW_VRAM_MODE = globals().get('LOW_VRAM_MODE', (DEVICE.type == 'cuda' and total_vram_gb <= 16.5))
if LOW_VRAM_MODE:
    ATN_BASE_CH = 24
    ATN_MAX_CH = 192
else:
    ATN_BASE_CH = 32
    ATN_MAX_CH = 256

atn = ResidualUNetATN(eps=EPSILON, base_ch=ATN_BASE_CH, max_ch=ATN_MAX_CH).to(DEVICE)
total_p = sum(p.numel() for p in atn.parameters())
print(f'ATN profile    : {"low-vram" if LOW_VRAM_MODE else "standard"} (base={ATN_BASE_CH}, max={ATN_MAX_CH})')
print(f'ATN parameters : {total_p:,}')
print(f'ATN device     : {next(atn.parameters()).device}')


In [ ]:
# Cell 5 - Surrogate Models + Frozen UFD Detector (detector-aware training)
LOW_VRAM_MODE = globals().get('LOW_VRAM_MODE', False)

surrogate_vgg = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)
for p in surrogate_vgg.parameters():
    p.requires_grad = False

if LOW_VRAM_MODE:
    # Start lean on memory-constrained sessions; Cell 7 can still fallback further if needed.
    surrogate_cas = None
    print('LOW_VRAM_MODE: loading only vggface2 surrogate at startup.')
else:
    surrogate_cas = InceptionResnetV1(pretrained='casia-webface').eval().to(DEVICE)
    for p in surrogate_cas.parameters():
        p.requires_grad = False

ufd_detector = None
if LOW_VRAM_MODE and DETECTOR_ENABLED:
    print('LOW_VRAM_MODE: detector disabled by default to preserve VRAM.')
    DETECTOR_ENABLED = False
    LAMBDA_DET = 0.0

if DETECTOR_ENABLED:
    # Kaggle dataset mounts + other common roots for UniversalFakeDetect
    ufd_candidates = [
        Path('/kaggle/input/universalfakedetect/UniversalFakeDetect'),
        Path('/kaggle/input/universalfakedetect'),
        Path('/kaggle/working/UniversalFakeDetect'),
        Path('/workspace/gitrepos/UniversalFakeDetect'),
        Path('/content/UniversalFakeDetect'),
        Path.cwd() / 'gitrepos' / 'UniversalFakeDetect',
    ]
    UFD_ROOT = next((p for p in ufd_candidates if (p / 'models').exists() or (p / 'models.py').exists()), None)
    if UFD_ROOT is None:
        print('WARNING: UniversalFakeDetect repo not found; detector loss disabled for this run.')
        DETECTOR_ENABLED = False
        LAMBDA_DET = 0.0
    else:
        sys.path.insert(0, str(UFD_ROOT))
        from models import get_model

        ufd_detector = get_model('CLIP:ViT-L/14')
        ufd_state = torch.load(DEEPSAFE_PTH, map_location=DEVICE)
        if isinstance(ufd_state, dict) and 'model' in ufd_state:
            ufd_state = ufd_state['model']
        incompat = ufd_detector.load_state_dict(ufd_state, strict=False)
        ufd_detector.to(DEVICE).eval()
        for p in ufd_detector.parameters():
            p.requires_grad_(False)

        print(f'UFD root    : {UFD_ROOT}')
        print(f'UFD detector: CLIP:ViT-L/14 on {next(ufd_detector.parameters()).device}')
        print(f'UFD state   : missing={len(incompat.missing_keys)} extra={len(incompat.unexpected_keys)}')
else:
    print('Detector-aware loss is disabled for this run.')

print(f'Surrogate 1 : vggface2 on {next(surrogate_vgg.parameters()).device}')
if surrogate_cas is None:
    print('Surrogate 2 : casia-webface NOT LOADED (low-vram startup)')
else:
    print(f'Surrogate 2 : casia-webface on {next(surrogate_cas.parameters()).device}')

if DEVICE.type == 'cuda':
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM used after loading models: {used:.2f} / {total:.1f} GB')


In [ ]:
# Cell 6 - Loss Functions
def perturbation_loss(pert):
    """Keep perturbation small and imperceptible: L2 + L_inf regularisation."""
    l2 = pert.pow(2).mean()
    linf = pert.abs().max(dim=1)[0].mean()
    return l2 + 0.1 * linf

def adversarial_loss(ce, se, margin=MARGIN):
    """Push shielded embedding away from clean embedding with hinge + distance term."""
    ce_n = F.normalize(ce, dim=1)
    se_n = F.normalize(se, dim=1)
    cos_sim = (ce_n * se_n).sum(dim=1)
    hinge = torch.clamp(cos_sim + margin, min=0).mean()
    l2_dist = F.pairwise_distance(ce_n, se_n).mean()
    return hinge - 0.5 * l2_dist

def detector_hinge_loss(det_logits):
    """Push fake logit below zero so shielded outputs look real to UFD detector."""
    return torch.clamp(det_logits, min=0).mean()

def total_loss(lx, ly, ldet):
    return LAMBDA_X * lx + LAMBDA_Y * ly + LAMBDA_DET * ldet

print(f'Losses ready. λx={LAMBDA_X} λy={LAMBDA_Y} λdet={LAMBDA_DET} margin={MARGIN}')


In [ ]:
# Cell 6b - Pre-compute Clean Embeddings
# Helper — re-usable so we can refresh the cache mid-training.
def build_clean_cache(loader, model):
    cache = {}
    model.eval()
    with torch.no_grad():
        for imgs, idxs in tqdm(loader, desc='Caching', leave=False):
            imgs = imgs.to(DEVICE, non_blocking=True)
            embs = F.normalize(model(imgs), dim=1).cpu()
            for emb, idx in zip(embs, idxs.tolist()):
                cache[idx] = emb
    return cache

print('Caching clean embeddings (vggface2)...')
clean_cache_vgg = build_clean_cache(cache_loader, surrogate_vgg)

if surrogate_cas is not None:
    print('Caching clean embeddings (casia-webface)...')
    clean_cache_cas = build_clean_cache(cache_loader, surrogate_cas)
    print(f'Cached {len(clean_cache_vgg)} embeddings per surrogate.')
else:
    clean_cache_cas = None
    print(f'Cached {len(clean_cache_vgg)} embeddings for vggface2 only.')

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
gc.collect()


In [ ]:
# Cell 7 - Training Loop (detector-aware objective enabled when available)
optimizer = torch.optim.AdamW(atn.parameters(), lr=LR, weight_decay=1e-4)
scaler = make_grad_scaler(USE_AMP)

# Warmup for first 5% of steps, then cosine anneal
WARMUP_EPOCHS = max(2, int(0.05 * EPOCHS))
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
history = {'lx': [], 'ly': [], 'ldet': [], 'total': [], 'embed_dist': [], 'epoch_time': []}

best_dist = -1.0
print(f'Starting training for {EPOCHS} epochs on {DEVICE}...')
print(f'Warmup epochs: {WARMUP_EPOCHS}')

# Adaptive micro-batching: keeps global BATCH_SIZE semantics but lowers peak VRAM.
if DEVICE.type == 'cuda':
    TRAIN_MICRO_BATCH = min(16, BATCH_SIZE)
else:
    TRAIN_MICRO_BATCH = BATCH_SIZE
print(f'Initial micro-batch size: {TRAIN_MICRO_BATCH}')

# Runtime fallback ladder for tight-memory GPUs
USE_SECOND_SURROGATE = (surrogate_cas is not None) and (clean_cache_cas is not None)
detector_fallback_done = False
surrogate_fallback_done = False

run_start = time.time()

for epoch in range(1, EPOCHS + 1):
    atn.train()
    rlx = rly = rldet = rtot = rdist = 0.0
    nb = 0
    t0 = time.time()

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False)
    for images, img_idxs in pbar:
        images = images.to(DEVICE, non_blocking=True)
        ce_vgg = torch.stack([clean_cache_vgg[i.item()] for i in img_idxs]).to(DEVICE, non_blocking=True)
        if USE_SECOND_SURROGATE:
            ce_cas = torch.stack([clean_cache_cas[i.item()] for i in img_idxs]).to(DEVICE, non_blocking=True)
        else:
            ce_cas = None

        batch_done = False
        while not batch_done:
            try:
                optimizer.zero_grad(set_to_none=True)

                bsz = images.size(0)
                step_lx = 0.0
                step_ly = 0.0
                step_ldet = 0.0
                step_total = 0.0

                # Chunk the batch for forward/backward to reduce activation memory.
                for s in range(0, bsz, TRAIN_MICRO_BATCH):
                    e = min(s + TRAIN_MICRO_BATCH, bsz)
                    w = (e - s) / bsz
                    x_mb = images[s:e]
                    ce_vgg_mb = ce_vgg[s:e]
                    ce_cas_mb = ce_cas[s:e] if ce_cas is not None else None

                    with amp_autocast(USE_AMP):
                        perturb = atn(x_mb)
                        shielded = torch.clamp(x_mb + perturb, -1.0, 1.0)

                        se_vgg = surrogate_vgg(shielded)
                        if USE_SECOND_SURROGATE:
                            se_cas = surrogate_cas(shielded)
                        else:
                            se_cas = None

                        if DETECTOR_ENABLED and ufd_detector is not None:
                            det_logit = ufd_detector(shielded)
                            ldet_mb = detector_hinge_loss(det_logit)
                        else:
                            ldet_mb = torch.zeros((), device=x_mb.device, dtype=x_mb.dtype)

                        lx_mb = perturbation_loss(perturb)
                        if USE_SECOND_SURROGATE:
                            ly_mb = 0.5 * (adversarial_loss(ce_vgg_mb, se_vgg) + adversarial_loss(ce_cas_mb, se_cas))
                        else:
                            ly_mb = adversarial_loss(ce_vgg_mb, se_vgg)
                        loss_mb = total_loss(lx_mb, ly_mb, ldet_mb) * w

                    scaler.scale(loss_mb).backward()

                    step_lx += float(lx_mb.detach().item()) * w
                    step_ly += float(ly_mb.detach().item()) * w
                    step_ldet += float(ldet_mb.detach().item()) * w
                    step_total += float(loss_mb.detach().item())

                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(atn.parameters(), max_norm=2.0)
                scaler.step(optimizer)
                scaler.update()

                # Distance metric on a small slice for low-overhead progress tracking.
                with torch.no_grad():
                    eval_n = min(8, bsz)
                    x_eval = images[:eval_n]
                    ce_eval = ce_vgg[:eval_n]
                    with amp_autocast(USE_AMP):
                        y_eval = torch.clamp(x_eval + atn(x_eval), -1.0, 1.0)
                        se_eval = surrogate_vgg(y_eval)
                    dist = (1 - F.cosine_similarity(
                        F.normalize(ce_eval, 2, 1),
                        F.normalize(se_eval, 2, 1))).mean().item()

                rlx += step_lx
                rly += step_ly
                rldet += step_ldet
                rtot += step_total
                rdist += dist
                nb += 1

                pbar.set_postfix(
                    Lx=f'{step_lx:.5f}',
                    Ly=f'{step_ly:.4f}',
                    Ldet=f'{step_ldet:.4f}',
                    Loss=f'{step_total:.4f}',
                    D=f'{dist:.4f}',
                    MB=TRAIN_MICRO_BATCH,
                    CAS='on' if USE_SECOND_SURROGATE else 'off',
                    DET='on' if DETECTOR_ENABLED else 'off'
                )

                batch_done = True

            except torch.cuda.OutOfMemoryError:
                if DEVICE.type != 'cuda':
                    raise

                torch.cuda.empty_cache()
                gc.collect()

                if TRAIN_MICRO_BATCH > 1:
                    old_mb = TRAIN_MICRO_BATCH
                    TRAIN_MICRO_BATCH = max(1, TRAIN_MICRO_BATCH // 2)
                    print(f'CUDA OOM: reducing micro-batch {old_mb} -> {TRAIN_MICRO_BATCH} and retrying batch...')
                    continue

                # Rescue step 1: disable detector loss and free detector VRAM.
                if DETECTOR_ENABLED and (ufd_detector is not None) and not detector_fallback_done:
                    print('CUDA OOM at micro-batch=1: disabling detector loss for this run and retrying...')
                    DETECTOR_ENABLED = False
                    LAMBDA_DET = 0.0
                    try:
                        ufd_detector = ufd_detector.to('cpu')
                    except Exception:
                        pass
                    torch.cuda.empty_cache()
                    gc.collect()
                    detector_fallback_done = True
                    continue

                # Rescue step 2: switch to single-surrogate mode.
                if USE_SECOND_SURROGATE and not surrogate_fallback_done:
                    print('CUDA OOM persists: switching to single-surrogate mode (vggface2 only) and retrying...')
                    USE_SECOND_SURROGATE = False
                    try:
                        if surrogate_cas is not None:
                            surrogate_cas = surrogate_cas.to('cpu')
                    except Exception:
                        pass
                    torch.cuda.empty_cache()
                    gc.collect()
                    surrogate_fallback_done = True
                    continue

                raise

    scheduler.step()
    et = time.time() - t0

    elx = rlx / nb
    ely = rly / nb
    eldet = rldet / nb
    etot = rtot / nb
    edist = rdist / nb
    history['lx'].append(elx)
    history['ly'].append(ely)
    history['ldet'].append(eldet)
    history['total'].append(etot)
    history['embed_dist'].append(edist)
    history['epoch_time'].append(et)

    lr_now = scheduler.get_last_lr()[0]
    print(
        f'Epoch {epoch:02d} | Lx={elx:.5f} Ly={ely:.5f} Ldet={eldet:.5f} '
        f'Total={etot:.5f} EmbDist={edist:.4f} LR={lr_now:.2e} Time={et:.0f}s | MB={TRAIN_MICRO_BATCH} '
        f'| CAS={"on" if USE_SECOND_SURROGATE else "off"} | DET={"on" if DETECTOR_ENABLED else "off"}'
    )

    if edist > best_dist:
        best_dist = edist
        torch.save(
            {
                'epoch': epoch,
                'model_state_dict': atn.state_dict(),
                'embed_dist': best_dist,
                'history': history,
            },
            BEST_PATH,
        )
        print(f'  * New best embedding distance {best_dist:.4f} -> saved to {BEST_PATH.name}')

    if epoch % 5 == 0:
        ckpt = CHECKPOINT_DIR / f'atn_epoch_{epoch:02d}.pth'
        torch.save(
            {
                'epoch': epoch,
                'model_state_dict': atn.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history,
            },
            ckpt,
        )
        print(f'  Checkpoint saved -> {ckpt.name}')

    if epoch % CACHE_REFRESH_EVERY == 0 and epoch < EPOCHS:
        print(f'  Refreshing embedding cache at epoch {epoch}...')
        clean_cache_vgg = build_clean_cache(cache_loader, surrogate_vgg)
        if USE_SECOND_SURROGATE and surrogate_cas is not None:
            clean_cache_cas = build_clean_cache(cache_loader, surrogate_cas)

    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

total_time = time.time() - run_start
print(f'\nTraining complete in {total_time/60:.1f} min')
print(f'Best embedding distance achieved: {best_dist:.4f}')

print('Loading best weights for evaluation...')
best_ckpt = torch.load(BEST_PATH, map_location=DEVICE)
atn.load_state_dict(best_ckpt['model_state_dict'])
print(f'Loaded best model from epoch {best_ckpt["epoch"]} (dist={best_ckpt["embed_dist"]:.4f})')


In [ ]:
# Cell 8 - Training Curves
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Perturbation Loss (Lx)', 'Adversarial Loss (Ly)',
          'Total Loss', 'Embedding Distance']
keys   = ['lx', 'ly', 'total', 'embed_dist']
colors = ['steelblue', 'tomato', 'mediumpurple', 'seagreen']
ep     = range(1, len(history['lx'])+1)
for ax, title, key, color in zip(axes, titles, keys, colors):
    ax.plot(ep, history[key], marker='o', color=color, linewidth=2)
    ax.set_title(title, fontsize=11); ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Best embedding distance: {max(history["embed_dist"]):.4f} '
      f'(epoch {history["embed_dist"].index(max(history["embed_dist"]))+1})')


In [ ]:
# Cell 9 - Visual Validation
atn.eval()
samples = val_images[:5]
fig, axes = plt.subplots(len(samples), 3, figsize=(12, 3*len(samples)))
if len(samples)==1: axes = axes[None]

for i, p in enumerate(samples):
    pil = Image.open(p).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    x   = val_tf(pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pert = atn(x)
        y    = torch.clamp(x + pert, -1.0, 1.0)

    def to_img(t): return ((t[0].cpu().permute(1,2,0).numpy()*0.5)+0.5).clip(0,1)
    xi = to_img(x); yi = to_img(y)
    pi = pert[0].cpu().permute(1,2,0).numpy()*10
    pi = (pi-pi.min())/(pi.max()-pi.min()+1e-8)

    # PSNR avoids scipy/skimage dependency issues on Kaggle.
    mse = float(np.mean((xi - yi) ** 2))
    psnr = 99.0 if mse == 0.0 else float(20.0 * np.log10(1.0 / np.sqrt(mse)))
    print(f'{p.name}: PSNR={psnr:.2f} dB')

    axes[i,0].imshow(xi);  axes[i,0].set_title('Original');          axes[i,0].axis('off')
    axes[i,1].imshow(pi);  axes[i,1].set_title('Perturbation x10');  axes[i,1].axis('off')
    axes[i,2].imshow(yi);  axes[i,2].set_title('Shielded');          axes[i,2].axis('off')

plt.tight_layout(); plt.show()


In [ ]:
# Cell 10 - Embedding Distance Validation
atn.eval()
distances_vgg = []
distances_cas = []

USE_CAS_VALIDATION = surrogate_cas is not None
if not USE_CAS_VALIDATION:
    print('CASIA surrogate not loaded (low-vram mode); validating with vggface2 only.')

with torch.no_grad():
    for p in tqdm(val_images[:100], desc='Validating'):
        x = val_tf(Image.open(p).convert('RGB').resize((IMG_SIZE, IMG_SIZE))).unsqueeze(0).to(DEVICE)
        pert = atn(x)
        y = torch.clamp(x + pert, -1.0, 1.0)

        e1v = F.normalize(surrogate_vgg(x), dim=1)
        e2v = F.normalize(surrogate_vgg(y), dim=1)
        distances_vgg.append((1 - F.cosine_similarity(e1v, e2v)).item())

        if USE_CAS_VALIDATION:
            e1c = F.normalize(surrogate_cas(x), dim=1)
            e2c = F.normalize(surrogate_cas(y), dim=1)
            distances_cas.append((1 - F.cosine_similarity(e1c, e2c)).item())

arr_vgg = np.array(distances_vgg)
print(f'[vggface2] Mean dist: {arr_vgg.mean():.4f}  Std: {arr_vgg.std():.4f}  '
      f'% > 0.5: {(arr_vgg > 0.5).mean()*100:.1f}%  (target mean > 0.30)')

if USE_CAS_VALIDATION and len(distances_cas) > 0:
    arr_cas = np.array(distances_cas)
    print(f'[casia-webface] Mean dist: {arr_cas.mean():.4f}  Std: {arr_cas.std():.4f}  '
          f'% > 0.5: {(arr_cas > 0.5).mean()*100:.1f}%  (target mean > 0.30)')

plt.figure(figsize=(8, 4))
plt.hist(arr_vgg, bins=30, color='steelblue', edgecolor='white')
plt.axvline(arr_vgg.mean(), color='red', linestyle='--', label=f'Mean={arr_vgg.mean():.3f}')
plt.axvline(0.30, color='orange', linestyle=':', label='Target=0.30')
plt.title('Embedding Distance Distribution (vggface2)')
plt.xlabel('Cosine Distance')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# Cell 11 - Export Model + Download
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
# Save the current (best) model weights
torch.save(atn.state_dict(), OUTPUT_PATH)

h = hashlib.sha256()
with open(OUTPUT_PATH,'rb') as f:
    for chunk in iter(lambda: f.read(1<<20), b''): h.update(chunk)
sha = h.hexdigest()
size_mb = OUTPUT_PATH.stat().st_size / (1024**2)

print(f'Saved  : {OUTPUT_PATH}')
print(f'Size   : {size_mb:.1f} MB')
print(f'SHA256 : {sha}')
print(f'Best embedding distance : {best_dist:.4f}')

if PLATFORM == 'colab':
    from google.colab import files
    print('Starting browser download...')
    files.download(str(OUTPUT_PATH))
    files.download(str(BEST_PATH))

elif PLATFORM == 'kaggle':
    print()
    print('On Kaggle: go to OUTPUT tab → download reface_atn.pth  and  reface_atn_best.pth')

elif PLATFORM == 'runpod':
    print()
    print('On RunPod use:')
    print(f'  scp root@<pod-ip>:{OUTPUT_PATH} .')

print('\nDone ✓')
